# Cloud Task Placement: Optimization Dashboard
**Design & Analysis of Algorithms (DAA) PBL — Group 3**  
*Mohak Gupta (`25WU0101074`), Naman Singla (`25WU0101084`), Mohammed Amaan Khan (`25WU0101075`)*

---

### Dashboard Overview
This interactive dashboard lets you configure parameters, execute the optimization pipeline, and visualize:
1. **MST Backbone Extraction (Kruskal/Prim)** for minimum-latency inter-server synchronization.
2. **All-Pairs Shortest Path Routing (Floyd-Warshall)** for end-to-end client access delays.
3. **0-1 Knapsack Dynamic Programming** for capacity-constrained placement selection.
4. **Visual Analytics**: Interactive topology plots, capacity utilization bars, cost-benefit trade-offs, and KPI metrics.

In [ ]:
# 1. Environment & Path Initialization
import os
import sys
import matplotlib.pyplot as plt

# Add project root to sys.path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) == 'notebooks' else notebook_dir
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.graph.graph import Graph
from src.graph.generator import generate_cloud_topology
from src.optimizer.workload import Task, generate_synthetic_workload
from src.optimizer.placement_optimizer import CloudPlacementOptimizer

print(" [OK] Core optimizer modules imported successfully!")

In [ ]:
# ==============================================================================
# 2. INTERACTIVE CONFIGURATION PARAMETERS
# Change these values and re-run all cells below to explore different scenarios!
# ==============================================================================

# Dataset choice: 'small' (10 servers), 'medium' (50 servers), 'large' (200 servers), or 'synthetic'
DATASET_CHOICE = 'small'

# Capacity scaling multiplier: e.g. 0.5 (constrained), 1.0 (standard), 2.0 (abundant)
CAPACITY_MULTIPLIER = 1.0

# Workload configuration
NUM_TASKS = 15                          # Total incoming tasks / microservice replicas
TASK_DEMAND_RANGE = (15, 45)            # Compute/RAM resource demand per task
CLIENT_TRAFFIC_RANGE = (25.0, 150.0)    # Client request traffic volume per task
RANDOM_SEED = 42                        # Seed for workload reproducibility

# Synthetic generator settings (used only if DATASET_CHOICE == 'synthetic')
SYNTHETIC_NODES = 20
SYNTHETIC_DENSITY = 0.25

print(f"Configuration loaded: Dataset='{DATASET_CHOICE}', Capacity Multiplier={CAPACITY_MULTIPLIER}x, Tasks={NUM_TASKS}")

In [ ]:
# 3. Run Optimization Pipeline
if DATASET_CHOICE in ('small', 'medium', 'large'):
    data_path = os.path.join(project_root, 'data', f'{DATASET_CHOICE}_topology.json')
    if os.path.exists(data_path):
        base_graph = Graph.load_from_json(data_path)
    else:
        base_graph = generate_cloud_topology(10, 0.2, seed=RANDOM_SEED)
else:
    base_graph = generate_cloud_topology(
        num_nodes=SYNTHETIC_NODES,
        edge_density=SYNTHETIC_DENSITY,
        latency_range=(2.0, 40.0),
        seed=RANDOM_SEED,
    )

# Scale server capacities according to parameter
graph = Graph(directed=False)
for node_id in base_graph.get_nodes():
    cap = max(1, int(base_graph.get_capacity(node_id) * CAPACITY_MULTIPLIER))
    graph.add_node(node_id, capacity=cap, **base_graph.get_node(node_id).metadata)
for u, v, lat in base_graph.get_edges():
    graph.add_edge(u, v, latency=lat)

# Ingest workload
tasks = generate_synthetic_workload(
    graph=graph,
    num_tasks=NUM_TASKS,
    demand_range=TASK_DEMAND_RANGE,
    requests_per_task=CLIENT_TRAFFIC_RANGE,
    seed=RANDOM_SEED,
)

# Execute end-to-end optimizer
optimizer = CloudPlacementOptimizer(graph, tasks)
report = optimizer.optimize()

print(f"Optimization complete for {graph.num_nodes} servers and {len(tasks)} tasks!")

In [ ]:
# 4. Executive KPI Metrics Card
print("=" * 78)
print("                   EXECUTIVE PERFORMANCE METRICS")
print("=" * 78)
print(f"  Baseline Average Client Latency   : {report.baseline_average_latency:6.2f} ms")
print(f"  Greedy Baseline Average Latency   : {report.greedy_optimized_latency:6.2f} ms (-{report.greedy_latency_reduction_percent:.1f}%)")
print(f"  Optimal DP Average Client Latency : {report.optimized_average_latency:6.2f} ms (-{report.latency_reduction_percent:.1f}%)")
print(f"  Fleet Capacity Utilization        : {report.total_capacity_utilization:6.1f} %")
print(f"  Tasks Placed vs Ingested          : {len(report.cost_benefit_table):2d} / {len(tasks):2d}")
print(f"  MST Backbone Total Latency        : {report.mst_total_weight:6.2f} ms ({len(report.mst_edges)} links)")
print(f"  Inter-Replica Backbone Sync Delay : {report.backbone_sync_latency:6.2f} ms")
print("=" * 78)

In [ ]:
# 5. Visualization 1: Topology & MST Backbone Overlay
plt.figure(figsize=(10, 8))
ax = plt.gca()

pos = {}
for node_id in graph.get_nodes():
    node = graph.get_node(node_id)
    pos[node_id] = (
        node.metadata.get('x', (node_id * 10) % 100),
        node.metadata.get('y', (node_id * 23) % 100),
    )

mst_set = {tuple(sorted((u, v))) for u, v, _ in report.mst_edges}

# Physical mesh links
for u, v, lat in graph.get_edges():
    if tuple(sorted((u, v))) not in mst_set:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        ax.plot([x1, x2], [y1, y2], color='#cccccc', linestyle='--', linewidth=1.2, zorder=1)

# MST Backbone links
for u, v, lat in report.mst_edges:
    x1, y1 = pos[u]
    x2, y2 = pos[v]
    ax.plot([x1, x2], [y1, y2], color='#0055d4', linestyle='-', linewidth=3.0, zorder=2)
    mid_x, mid_y = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    ax.text(mid_x, mid_y, f"{lat:.1f}ms", fontsize=8, color='#003380', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#e6f0ff', edgecolor='none', alpha=0.85))

servers_with_tasks = {s.server_id for s in report.server_summaries if s.placed_tasks}

# Server nodes
for n in graph.get_nodes():
    x, y = pos[n]
    cap = graph.get_capacity(n)
    is_host = n in servers_with_tasks
    color = '#2ca02c' if is_host else '#ff7f0e'
    size = max(180, cap * 3.0)
    ax.scatter([x], [y], s=size, color=color, edgecolors='#333333', linewidth=1.5, zorder=3)
    ax.text(x, y, f"S{n}", fontsize=9, fontweight='bold', color='white', ha='center', va='center', zorder=4)

ax.set_title(
    f"Cloud Network Topology & MST Backbone Overlay\nTotal Backbone Weight: {report.mst_total_weight:.2f} ms ({len(report.mst_edges)} links)",
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('X Coordinate', fontsize=11)
ax.set_ylabel('Y Coordinate', fontsize=11)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], color='#0055d4', lw=3, label='MST Backbone Link'),
    Line2D([0], [0], color='#cccccc', lw=1.2, linestyle='--', label='Physical Mesh Link'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ca02c', markersize=10, label='Server Hosting Placed Tasks'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff7f0e', markersize=10, label='Idle Server (Available Capacity)'),
]
ax.legend(handles=legend_handles, loc='upper right', framealpha=0.92, fontsize=9)
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Visualization 2: Server Capacity Allocation & Task Distribution
servers = [s.server_id for s in report.server_summaries]
caps = [s.total_capacity for s in report.server_summaries]
used = [s.used_capacity for s in report.server_summaries]

fig, ax = plt.subplots(figsize=(12, 5))
indices = list(range(len(servers)))
bar_width = 0.55

ax.bar(indices, caps, width=bar_width, label='Remaining Capacity', color='#e0e0e0', edgecolor='#999999')
ax.bar(indices, used, width=bar_width, label='Allocated Capacity (Placed Tasks)', color='#1b9e77', edgecolor='#0e533e')

for i, s in enumerate(report.server_summaries):
    util_str = f"{s.utilization_percent:.0f}%"
    tasks_count = f"{len(s.placed_tasks)} tasks" if s.placed_tasks else "idle"
    ax.text(
        i, s.total_capacity + max(caps) * 0.02,
        f"{util_str}\n({tasks_count})",
        ha='center', va='bottom', fontsize=8, fontweight='bold'
    )

ax.set_xticks(indices)
ax.set_xticklabels([f"Server {s}" for s in servers], rotation=0)
ax.set_ylabel('Compute Capacity Units (CPU/RAM)', fontsize=11)
ax.set_title('Server Capacity Utilization & Task Allocations (0-1 Knapsack DP)', fontsize=13, fontweight='bold', pad=12)
ax.set_ylim(0, max(caps) * 1.25)
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Visualization 3: Placed Tasks Cost vs Latency Benefit Trade-off
if report.cost_benefit_table:
    costs = [d.demand_cost for d in report.cost_benefit_table]
    unit_bens = [d.latency_benefit for d in report.cost_benefit_table]
    total_bens = [d.total_benefit for d in report.cost_benefit_table]
    task_labels = [d.task_id for d in report.cost_benefit_table]
    hosts = [f"S{d.server_id}" for d in report.cost_benefit_table]

    fig, ax = plt.subplots(figsize=(10, 5))
    scatter = ax.scatter(costs, unit_bens, c=total_bens, cmap='viridis', s=180, edgecolors='black', linewidth=1.2, zorder=3)
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Total Weighted Benefit (ms * req/s)', fontsize=10)

    for i, txt in enumerate(task_labels):
        ax.annotate(f"{txt} -> {hosts[i]}", (costs[i] + 0.3, unit_bens[i] + 0.3), fontsize=8, fontweight='bold')

    ax.set_xlabel('Resource Demand Cost (CPU/RAM Units)', fontsize=11)
    ax.set_ylabel('Per-Request Latency Reduction (ms)', fontsize=11)
    ax.set_title('Placed Tasks: Resource Cost vs Latency Reduction Benefit', fontsize=13, fontweight='bold', pad=12)
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("No tasks placed under current constraints.")

In [ ]:
# 8. Placed Tasks Breakdown Table
print(f"{'Task ID':<9} | {'Host':<8} | {'Cost':<5} | {'Base Lat':<8} | {'Opt Lat':<8} | {'Unit Ben':<8} | {'Total Benefit':<13} | {'Efficiency'}")
print("-" * 82)
for d in report.cost_benefit_table:
    print(
        f"{d.task_id:<9} | "
        f"Server {d.server_id:<1} | "
        f"{d.demand_cost:<5} | "
        f"{d.baseline_latency:8.2f} | "
        f"{d.optimized_latency:8.2f} | "
        f"{d.latency_benefit:8.2f} | "
        f"{d.total_benefit:13.1f} | "
        f"{d.efficiency_ratio:8.4f}"
    )